<a href="https://colab.research.google.com/github/revak18/Hurricane_Risk_Assessment/blob/main/HazardGraph_targets_pipeline_runner.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# HazardGraph : R `targets` pipeline runner

This notebook runs the HazardGraph R targets pipeline in Google Colab using scripts and data stored in Google Drive.

Steps: Configure → Setup → Download scripts → Verify → Run → Review results.

## 1. Configuration
Project locations. Scientific settings (state, hurricane, buffers) are in 00_config.R.

In [6]:
drive_base <- "Science_Fair/ScienceFairProject2027/HazardGraph"
project_dir <- "/content/hazardgraph_pipeline"
local_base <- "/content/hazardgraph"
package_archive <- "/content/colab_r_libraries.zip"
user_lib <- file.path(Sys.getenv("HOME"), "R", "local-library")
dir.create(project_dir, recursive = TRUE, showWarnings = FALSE)
dir.create(user_lib, recursive = TRUE, showWarnings = FALSE)
.libPaths(c(user_lib, .libPaths()))

## 2. Google Drive and R packages
Authenticate Drive. Restore the saved R library only if amadeus is missing.

In [7]:
options(repos = c(CRAN = "https://cloud.r-project.org"))
.libPaths(c(user_lib, .libPaths()))

if (!requireNamespace("googledrive", quietly = TRUE))
  install.packages("googledrive", lib = user_lib)
library(googledrive)
if (!drive_has_token()) drive_auth(use_oob = TRUE)

if (!requireNamespace("amadeus", quietly = TRUE)) {
  file <- drive_get(file.path(drive_base, "Colab_R_libraries", "colab_r_libraries.zip"))
  drive_download(file, package_archive, overwrite = TRUE)
  unzip(package_archive, exdir = "/")
  .libPaths(c(user_lib, .libPaths()))
}

packages <- c("targets", "readr", "terra", "sf", "ggplot2", "tidyselect")
missing <- packages[!vapply(packages, requireNamespace, logical(1), quietly = TRUE)]
if (length(missing)) install.packages(missing, lib = user_lib)
stopifnot(all(vapply(c("amadeus", packages), requireNamespace, logical(1), quietly = TRUE)))
library(targets)
message("Setup complete.")

Setup complete.



## 3. Google Drive folders
Create missing project folders and save their IDs for the R scripts.

In [8]:
folder <- function(parent, name) {
  x <- drive_ls(parent, type = "folder")
  x <- x[x$name == name, ]
  if (nrow(x) > 1) stop("Duplicate folder: ", name)
  if (!nrow(x)) x <- drive_mkdir(name, path = parent)
  x
}
folder_ids <- function(parent, names, prefix = "")
  setNames(lapply(names, function(n) folder(parent, n)$id[[1]]), paste0(prefix, names))

project <- drive_get(path = drive_base)
stopifnot(nrow(project) == 1)
scripts_folder <- folder(project, "scripts")
output <- folder(project, "Amadeus_Cloud_Downloads")
raw <- folder(output, "raw")
processed <- folder(output, "processed")

groups <- c("tri", "cdl", "geography", "mines", "hurricanes", "livestock",
            "superfund", "brownfields", "wastewater", "rmp", "substations", "hospitals")
drive_folders <- c(
  folder_ids(raw, groups, "raw_"),
  folder_ids(processed, c(groups, "exposure_features"), "processed_"),
  folder_ids(output, c("metadata", "validation", "figures"))
)
saveRDS(drive_folders, file.path(project_dir, "drive_folders.rds"))
message("Drive folders ready.")

Auto-refreshing stale OAuth token.

✔ The input `path` resolved to exactly 1 file.

Drive folders ready.



## 4. Download pipeline scripts
Download _targets.R and the scripts it references from Google Drive.

In [9]:
scripts <- drive_ls(scripts_folder)
download_script <- function(name) {
  file <- scripts[scripts$name == name, ]
  if (nrow(file) != 1) stop("Missing or duplicate script: ", name)
  drive_download(file, file.path(project_dir, name), overwrite = TRUE)
}

download_script("_targets.R")
lines <- readLines(file.path(project_dir, "_targets.R"), warn = FALSE)
sources <- grep('^[[:space:]]*source\\("', lines, value = TRUE)
files <- unique(sub('.*source\\("([^"]+)"\\).*', '\\1', sources))
if (!length(files)) stop("No source scripts found.")
invisible(lapply(files, download_script))
message("Downloaded ", length(files) + 1, " scripts.")

File downloaded:

• _targets.R <id: 1AhZjJzzt7RrHvu_6RmGjOFBsEdT_A1Jl>

Saved locally as:

• /content/hazardgraph_pipeline/_targets.R

File downloaded:

• 00_config.R <id: 1J1edoSMnMKmLe0GwlzVfnL2rXuqqwx1o>

Saved locally as:

• /content/hazardgraph_pipeline/00_config.R

File downloaded:

• 01_tri.R <id: 1Jm3uSi6Hk7pG6qy5S8iPH1Cr4GNw3Sy7>

Saved locally as:

• /content/hazardgraph_pipeline/01_tri.R

File downloaded:

• 02_cdl.R <id: 1Lap1Ey9wwSWtglC9AiMd0SOLXHUDYVSa>

Saved locally as:

• /content/hazardgraph_pipeline/02_cdl.R

File downloaded:

• 03_qaqc.R <id: 1JWRNV4Gg8IHaabbcaJsG9oWi1f9S0ErF>

Saved locally as:

• /content/hazardgraph_pipeline/03_qaqc.R

File downloaded:

• 04_geography.R <id: 1z4wJHFajGY7z2B1LXyzLkQtSGZS2bIpw>

Saved locally as:

• /content/hazardgraph_pipeline/04_geography.R

File downloaded:

• 05_tri_fips.R <id: 1grlhvBdIvA8zjdtnEzn61f8zB7s7eS2I>

Saved locally as:

• /content/hazardgraph_pipeline/05_tri_fips.R

File downloaded:

• 06_tri_geography_qaqc.R <id: 

## 5. Verify the pipeline definition

Confirm all scripts can be parsed and inspect the registered targets **before** running anything expensive.

In [5]:
# Verify scripts and targets
setwd(project_dir)
stopifnot(all(file.exists(c("_targets.R", files, "drive_folders.rds"))))

manifest <- targets::tar_manifest()
cat("Verified", length(files) + 1, "scripts and", nrow(manifest), "targets.\n")

Hurricane Ian settings ready.
Hurricane Ian exposure settings ready.
Project settings ready.
Local project folders ready.
Google Drive folder IDs loaded.
Google Drive pipeline folders ready.
Pipeline files loaded.
Pipeline packages registered.
Pipeline paths ready.
Verified 44 scripts and 94 targets.


## 6. Run the full pipeline (optional, off by default)
Set run_full <- TRUE only when you intend to update the full pipeline. A fresh Colab runtime may rebuild earlier work if the local targets cache is unavailable.

In [ ]:
run_full <- FALSE
if (run_full) {
  setwd(project_dir)
  targets::tar_make(callr_function = NULL)
}

## 7. Check target status
Read local targets status and error records. No data processing is executed.

In [10]:
setwd(project_dir)
outdated <- targets::tar_outdated()
cat("Outdated targets:", length(outdated), "\n")
if (length(outdated)) print(head(outdated, 20))
meta <- targets::tar_meta(fields = c(name, seconds, error))
failed <- meta[!is.na(meta$error) & nzchar(as.character(meta$error)), ]
if (nrow(failed)) print(failed[, c("name", "error")], row.names = FALSE)

Hurricane Ian settings ready.
Hurricane Ian exposure settings ready.
Project settings ready.
Local project folders ready.
Google Drive folder IDs loaded.
Google Drive pipeline folders ready.
Pipeline files loaded.
Pipeline packages registered.
Pipeline paths ready.
Outdated targets: 94 
 [1] "ian_track_files"                     
 [2] "nc_counties_raw"                     
 [3] "hazardgraph_substation_upload"       
 [4] "hazardgraph_rmp_upload"              
 [5] "brownfields_proximity_upload"        
 [6] "hospital_proximity_upload"           
 [7] "rmp_raw_file"                        
 [8] "mrds_schema_files"                   
 [9] "ian_track_upload"                    
[10] "mrds_proximity_files"                
[11] "statewide_hazardgraph_analysis_files"
[12] "wastewater_files"                    
[13] "qaqc_report"                         
[14] "wastewater_proximity_files"          
[15] "tri_geography_qaqc_files"            
[16] "superfund_files"                     
[17] "ha

No local metadata. Did you remember to run tar_meta_download()?
Details: https://books.ropensci.org/targets/cloud-storage.html.
(Or maybe you need to run the pipeline with tar_make()?)
Silence this message with Sys.setenv(TAR_WARN = "false").


## 8. Check existing Google Drive files (read-only)
Shows saved raw inputs, processed datasets, reports, and figures, including their last-modified dates. This does not run the pipeline or download data. Drive output presence is not proof that a target completed successfully.

In [11]:
# Inventory all configured Drive data/output folders
parts <- lapply(names(drive_folders), function(group) {
  x <- drive_ls(as_id(drive_folders[[group]]))
  if (!nrow(x)) return(NULL)
  updated <- vapply(x$drive_resource, function(z)
    if (is.null(z$modifiedTime)) "" else z$modifiedTime, character(1))
  data.frame(folder = group, name = x$name, id = x$id, updated = updated,
             link = paste0("https://drive.google.com/open?id=", x$id))
})
inventory <- do.call(rbind, parts)
if (is.null(inventory)) stop("No files found in the configured Drive folders.")

by_folder <- split(inventory, inventory$folder)
summary <- data.frame(folder = names(by_folder),
  files = vapply(by_folder, nrow, integer(1)),
  last_updated = vapply(by_folder, function(x) max(x$updated), character(1)))
print(summary, row.names = FALSE)

results <- inventory[inventory$folder %in% c("processed_exposure_features", "validation", "figures"), ]
results <- results[order(results$updated, decreasing = TRUE), ]
cat("\nMost recent saved results:\n")
print(head(results[, c("folder", "name", "updated", "link")], 20), row.names = FALSE)
write.csv(inventory, file.path(project_dir, "drive_inventory.csv"), row.names = FALSE)

                      folder files             last_updated
                     figures    12 2026-10-18T18:12:24.966Z
                    metadata     9 2026-09-01T03:38:51.876Z
       processed_brownfields     4 2026-08-31T01:54:15.612Z
               processed_cdl     3 2026-08-30T22:37:29.709Z
 processed_exposure_features    42 2026-10-18T18:23:36.131Z
         processed_geography     1 2026-08-29T23:31:31.889Z
         processed_hospitals     4 2026-09-01T03:38:49.212Z
        processed_hurricanes     8 2026-08-30T22:08:33.909Z
         processed_livestock     3 2026-08-31T00:43:01.885Z
             processed_mines     5 2026-09-14T01:23:09.853Z
       processed_substations     4 2026-09-01T02:51:17.964Z
         processed_superfund     4 2026-08-31T01:23:20.460Z
               processed_tri     7 2026-08-30T01:57:02.494Z
        processed_wastewater     4 2026-08-31T02:33:44.415Z
             raw_brownfields     5 2026-10-18T14:52:38.636Z
               raw_geography     1 2026-

## 9. Quick test and mentor outputs (optional)
Target test: Change target_name, then set run_target <- TRUE. A selected target can still rebuild its upstream dependencies. This cell blocks execution if the local targets cache is absent.

Export: Pick file names shown in Step 8 to create a small ZIP of existing Drive outputs. Exporting does not run the pipeline.

In [ ]:
# Run one selected target only when needed
run_target <- FALSE
target_name <- ""  # Choose a name from tar_manifest()$name
if (run_target) {
  setwd(project_dir)
  if (!file.exists("_targets/meta/meta")) stop("Local targets cache missing; upstream work may rerun.")
  stopifnot(target_name %in% targets::tar_manifest()$name)
  targets::tar_make(names = tidyselect::all_of(target_name), callr_function = NULL)
}

In [ ]:
# Download only selected results and make a shareable ZIP
mentor_files <- character()  # Example: c("figure.png", "results.csv")
if (length(mentor_files)) {
  chosen <- inventory[inventory$name %in% mentor_files, ]
  if (nrow(chosen) != length(unique(mentor_files)) || anyDuplicated(chosen$name))
    stop("Each selected name must match exactly one Drive file.")
  out <- file.path(project_dir, "mentor_outputs")
  dir.create(out, showWarnings = FALSE)
  for (i in seq_len(nrow(chosen)))
    drive_download(as_id(chosen$id[i]), file.path(out, chosen$name[i]), overwrite = TRUE)
  write.csv(chosen[, c("folder", "name", "updated", "link")],
            file.path(out, "file_manifest.csv"), row.names = FALSE)
  setwd(out)
  utils::zip(file.path(project_dir, "HazardGraph_mentor_outputs.zip"),
             files = c(chosen$name, "file_manifest.csv"))
  setwd(project_dir)
  message("Created: ", file.path(project_dir, "HazardGraph_mentor_outputs.zip"))
}

Notes:
- Step 8 lists files in Google Drive even if Colab's _targets cache is gone.
- Step 9 does not run targets until explicitly enabled. It cannot guarantee an incremental run if upstream targets are outdated.
- After adding an R script or algorithm, update _targets.R and the Drive scripts/ copies, rerun Steps 4–5, then use Step 9 to test the selected target. Rerun Step 8 to refresh the inventory after new results are uploaded.
- Keep publication-ready figures and tables as pipeline outputs; select only the relevant files for the mentor ZIP.